# Model 2: Decision Tree Classifier
## IoT Saldırı Tipi Sınıflandırması (Multi-class)

Bu notebook, **Edge-IIoTset** veri seti üzerinde **Attack_type** kolonunu hedef değişken olarak kullanan
Decision Tree (Karar Ağacı) modelini adım adım eğitir ve değerlendirir.

### Neden Decision Tree?
- **Tam yorumlanabilirlik**: Ağaç yapısı insan tarafından okunabilir karar kuralları üretir
- **Feature scaling gerektirmez**: Ağaç tabanlı modeller scale'den bağımsızdır
- **Feature importance**: Gini/Entropy bazlı doğrudan önem skoru verir
- **Non-linear ilişkiler**: Doğrusal olmayan örüntüleri yakalayabilir

### Pipeline Adımları
1. Gold Delta Lake → Veri yükleme
2. Feature vektörleme + StringIndexer
3. Sınıf ağırlıkları (classWeight)
4. Stratified Train/Test split (%80/%20)
5. DecisionTreeClassifier (Gini/Entropy)
6. CrossValidator ile hiperparametre arama (maxDepth, minInstancesPerNode, impurity)
7. Test seti değerlendirme
8. Ağaç yapı analizi + Feature importance
9. MLflow loglama

## 1. Kütüphaneler ve Spark Session

In [ ]:
import sys
import time
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from pyspark.ml import Pipeline
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

sys.path.insert(0, "/opt/bitnami/spark")

from spark.spark_session import get_spark
from ml.utils import (
    run_ml_pipeline_multiclass,
    evaluate_model_multiclass,
    compute_confusion_matrix_multiclass,
    log_to_mlflow,
)

print("Kütüphaneler yüklendi.")

In [ ]:
spark = get_spark("Notebook-DecisionTree-Multiclass")
print(f"Spark version: {spark.version}")
print(f"App name: {spark.sparkContext.appName}")

## 2. Veri Yükleme ve Feature Hazırlığı

Gold katmanından çok sınıflı ML pipeline'ı çalıştırılır:
- StringIndexer ile Attack_type → sayısal label
- VectorAssembler ile numerik feature vektörü
- Balanced class weights
- Stratified split

In [ ]:
SAMPLE_SIZE = 30000  # Hızlı test için (None = tüm veri)

stage_start = time.perf_counter()
train_df, test_df, feature_cols, label_index_model = run_ml_pipeline_multiclass(
    spark,
    sample_size=SAMPLE_SIZE,
    split_log_stats=True,
)
label_names = list(label_index_model.labels)
num_classes = len(label_names)

print(f"\nPipeline süresi: {time.perf_counter() - stage_start:.2f}s")
print(f"Feature sayısı: {len(feature_cols)}")
print(f"Sınıf sayısı: {num_classes}")
print(f"Sınıflar: {label_names}")

In [ ]:
train_count = train_df.count()
test_count = test_df.count()
print(f"Train: {train_count:,} satır")
print(f"Test:  {test_count:,} satır")

print("\nTrain seti sınıf dağılımı:")
train_df.groupBy("label").count().orderBy("label").show()

## 3. Model Pipeline Oluşturma

Decision Tree'de scaling gerekmez. Doğrudan feature vektörü kullanılır.

### Hiperparametreler
| Parametre | Açıklama | Aranacak Değerler |
|-----------|----------|------------------|
| `maxDepth` | Ağacın maksimum derinliği | 5, 10, 15 |
| `minInstancesPerNode` | Yaprak düğüm min örnek sayısı | 1, 5 |
| `impurity` | Bölme kriteri | gini, entropy |

In [ ]:
dt = DecisionTreeClassifier(
    featuresCol="features",
    labelCol="label",
    weightCol="classWeight",
    maxDepth=10,
    seed=42,
)
pipeline = Pipeline(stages=[dt])
print("Pipeline oluşturuldu: DecisionTreeClassifier")

## 4. Cross Validation

In [ ]:
max_depth_values = [5, 10, 15]
min_instances_values = [1, 5]
impurity_values = ["gini", "entropy"]
num_folds = 2

param_grid = (
    ParamGridBuilder()
    .addGrid(dt.maxDepth, max_depth_values)
    .addGrid(dt.minInstancesPerNode, min_instances_values)
    .addGrid(dt.impurity, impurity_values)
    .build()
)

cv = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=param_grid,
    evaluator=MulticlassClassificationEvaluator(
        labelCol="label", predictionCol="prediction", metricName="f1",
    ),
    numFolds=num_folds,
    seed=42,
    parallelism=2,
)

total_cv_runs = len(param_grid) * num_folds
print(f"Grid boyutu: {len(param_grid)} kombinasyon")
print(f"Fold sayısı: {num_folds}")
print(f"Toplam fit: {total_cv_runs}")

## 5. Model Eğitimi

In [ ]:
print("Cross Validation başlatılıyor...")
cv_start = time.perf_counter()
cv_model = cv.fit(train_df)
cv_duration = time.perf_counter() - cv_start
print(f"CV tamamlandı! Süre: {cv_duration:.2f}s")

# En iyi model parametreleri
best_dt_model = cv_model.bestModel.stages[-1]
print(f"\nEn iyi parametreler:")
print(f"  maxDepth:            {best_dt_model.getOrDefault('maxDepth')}")
print(f"  minInstancesPerNode: {best_dt_model.getOrDefault('minInstancesPerNode')}")
print(f"  impurity:            {best_dt_model.getOrDefault('impurity')}")

In [ ]:
# CV sonuçları
print("CV Sonuçları (F1-Score):")
print(f"{'#':<4} {'maxDepth':<10} {'minInst':<10} {'impurity':<10} {'Avg F1':>10}")
print("-" * 48)
for i, (params, score) in enumerate(zip(param_grid, cv_model.avgMetrics)):
    param_dict = {p.name: v for p, v in params.items()}
    marker = " ← best" if score == max(cv_model.avgMetrics) else ""
    print(f"{i+1:<4} {param_dict.get('maxDepth', ''):<10} {param_dict.get('minInstancesPerNode', ''):<10} {param_dict.get('impurity', ''):<10} {score:>10.4f}{marker}")

## 6. Test Seti Değerlendirme

In [ ]:
predictions = cv_model.bestModel.transform(test_df)
metrics = evaluate_model_multiclass(predictions, num_classes=num_classes)

In [ ]:
confusion = compute_confusion_matrix_multiclass(predictions, label_names=label_names)

In [ ]:
print("\n" + "=" * 40)
print("SONUÇ ÖZETİ")
print("=" * 40)
print(f"Accuracy:  {metrics.get('accuracy', 0):.4f}")
print(f"F1-Score:  {metrics.get('f1_score', 0):.4f}")
print(f"Precision: {metrics.get('precision', 0):.4f}")
print(f"Recall:    {metrics.get('recall', 0):.4f}")

## 7. Ağaç Yapısı Analizi

Decision Tree'nin yorumlanabilirliğinin en güçlü yanı: ağaç yapısını doğrudan inceleyebilmek.

In [ ]:
# Ağaç yapısı bilgileri
depth = best_dt_model.depth
num_nodes = best_dt_model.numNodes
num_leaves = (num_nodes + 1) // 2

print(f"Ağaç Derinliği: {depth}")
print(f"Toplam Düğüm:   {num_nodes}")
print(f"Yaprak Sayısı:  ~{num_leaves}")

# Ağaç yapısının ilk 30 satırı
debug_string = best_dt_model.toDebugString
debug_lines = debug_string.split("\n")
print(f"\nKarar Ağacı Yapısı (ilk 30 satır):")
print("-" * 60)
for line in debug_lines[:30]:
    print(line)
if len(debug_lines) > 30:
    print(f"... ({len(debug_lines) - 30} satır daha)")

## 8. Feature Importance

In [ ]:
# Feature importance (Gini/Entropy bazlı)
importances = best_dt_model.featureImportances.toArray().tolist()
fi_pairs = sorted(zip(feature_cols, importances), key=lambda x: x[1], reverse=True)
top_features = fi_pairs[:15]

print("Top 15 Feature Importance:")
print(f"{'#':<4} {'Feature':<35} {'Importance':>12}")
print("-" * 55)
for idx, (fname, importance) in enumerate(top_features, start=1):
    print(f"{idx:<4} {fname:<35} {importance:>12.6f}")

In [ ]:
# Feature Importance Görselleştirmesi
top10 = fi_pairs[:10]
names = [f[0] for f in reversed(top10)]
values = [f[1] for f in reversed(top10)]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(names, values, color="#2196F3", edgecolor="#1565C0", height=0.6)
for bar, val in zip(bars, values):
    ax.text(bar.get_width() + max(values) * 0.01,
            bar.get_y() + bar.get_height() / 2,
            f"{val:.4f}", va="center", fontsize=9, fontweight="bold")

ax.set_xlabel("Feature Importance (Gini/Entropy)", fontsize=11)
ax.set_title("Decision Tree — Top 10 Feature Importance (Multi-class)",
             fontsize=13, fontweight="bold")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

## 9. MLflow'a Loglama

In [ ]:
best_params = {
    "maxDepth": int(best_dt_model.getOrDefault("maxDepth")),
    "minInstancesPerNode": int(best_dt_model.getOrDefault("minInstancesPerNode")),
    "impurity": str(best_dt_model.getOrDefault("impurity")),
    "numClasses": int(num_classes),
    "numFolds": num_folds,
    "grid_size": len(param_grid),
    "cv_total_fits": total_cv_runs,
    "weightCol": "classWeight",
    "tree_depth": depth,
    "tree_num_nodes": num_nodes,
    "tree_num_leaves": num_leaves,
    "label_column": "Attack_type",
}

metrics["tree_depth"] = float(depth)
metrics["tree_num_nodes"] = float(num_nodes)

confusion_metrics = {}
for i, name in enumerate(label_names):
    confusion_metrics[f"row_total_class_{i}"] = confusion["row_totals"][i]
    confusion_metrics[f"per_class_acc_{i}"] = confusion["per_class_acc"][i]

run_id = log_to_mlflow(
    run_name="decision_tree_notebook",
    model_type="DecisionTree",
    params=best_params,
    metrics={**metrics, **confusion_metrics},
    model=cv_model.bestModel,
    feature_importance=top_features[:10],
    tags={
        "source": "notebook",
        "model_index": "2",
        "classification_type": "multiclass",
        "interpretable": "true",
    },
)
print(f"\nMLflow Run ID: {run_id}")

## Özet

Bu notebook'ta **Decision Tree** modelini başarıyla eğittik:

- **Gini/Entropy** impurity kriteri ile bölme yapıldı
- **CrossValidator** ile maxDepth, minInstancesPerNode optimize edildi
- **classWeight** ile sınıf dengesizliği telafi edildi
- Ağaç yapısı (derinlik, düğüm sayısı) analiz edildi
- **Feature importance** Gini/Entropy bazlı hesaplandı
- Sonuçlar **MLflow**'a loglandı

Decision Tree, tamamen yorumlanabilir bir model olarak karar kurallarının incelenmesine olanak tanır.

In [ ]:
spark.stop()
print("Spark oturumu kapatıldı.")